# Dataset

In [57]:
import pandas as pd

# Nome das colunas do dataset
colunas = [
    'top_left', 'top_middle', 'top_right',
    'mid_left', 'mid_middle', 'mid_right',
    'bot_left', 'bot_middle', 'bot_right',
    'classe'
]

# 2. Ler o dataset original
df = pd.read_csv('/tic-tac-toe.data', names=colunas)

# 3. Exibir as primeiras linhas do dataset
df.head()

,top_left,top_middle,top_right,mid_left,mid_middle,mid_right,bot_left,bot_middle,bot_right,classe
0,x,x,x,x,o,o,x,o,o,positive
1,x,x,x,x,o,o,o,x,o,positive
2,x,x,x,x,o,o,o,o,x,positive
3,x,x,x,x,o,o,o,b,b,positive
4,x,x,x,x,o,o,b,o,b,positive


In [58]:
# Dicionário de mapeamento para converter os valores textuais do tabuleiro ('x', 'o', 'b') em valores numéricos (1, -1, 0).
# 'x' (jogador X) é mapeado para 1.
# 'o' (jogador O) é mapeado para -1.
# 'b' (espaço em branco) é mapeado para 0.
mapeamento = {'x': 1, 'o': -1, 'b': 0}
## Assim a gente converte em números os valores de texto das jogadas

# Aplicando o mapeamento em todas as colunas de posição do tabuleiro.
# Itera sobre as colunas
for col in df.columns[:-1]:
    df[col] = df[col].map(mapeamento)

# Verificando a transformação exibindo as primeiras linhas do DataFrame com os novos valores numéricos.
df.head()

,top_left,top_middle,top_right,mid_left,mid_middle,mid_right,bot_left,bot_middle,bot_right,classe
0,1,1,1,1,-1,-1,1,-1,-1,positive
1,1,1,1,1,-1,-1,-1,1,-1,positive
2,1,1,1,1,-1,-1,-1,-1,1,positive
3,1,1,1,1,-1,-1,-1,0,0,positive
4,1,1,1,1,-1,-1,0,-1,0,positive


In [59]:
import pandas as pd
import numpy as np


def reclassificar(row):
    # Pega 9 posições do tabuleiro, que correspondem às entradas do jogo.
    # 'seleciona as 9 primeiras colunas de cada linha, que representam o estado do tabuleiro.
    tab = row.iloc[:9].values.tolist()
    # Define as 8 combinações de vitória (3 linhas, 3 colunas, 2 diagonais) em um tabuleiro 3x3.
    combos = [
        [0,1,2], [3,4,5], [6,7,8], # linhas
        [0,3,6], [1,4,7], [2,5,8], # colunas
        [0,4,8], [2,4,6]           # diagonais
    ]

    # Verifica se o jogador 'O' (representado por -1) venceu.
    # Itera por todas as combinações de vitória e checa se todas as posições da combinação são -1.
    for c in combos:
        if tab[c[0]] == tab[c[1]] == tab[c[2]] == -1: return "O vence"

    # Verifica se o jogador 'X' (representado por 1) venceu.
    # Itera por todas as combinações de vitória e checa se todas as posições da combinação são 1.
    for c in combos:
        if tab[c[0]] == tab[c[1]] == tab[c[2]] == 1: return "X vence"

    # Se não há zeros (espaços vazios) no tabuleiro e ninguém venceu, o jogo é um empate.
    # '0 not in tab' verifica se ainda há posições vazias no tabuleiro.
    if 0 not in tab: return "Empate"

    # Se ainda tem espaços vazios (0 está em 'tab') e ninguém venceu, o jogo ainda está em andamento.
    return "Tem jogo"

# Cria a coluna 'classe_final' no DataFrame original aplicando a função 'reclassificar' a cada linha.
df['classe_final'] = df.apply(reclassificar, axis=1)

# Imprime a distribuição dos novos valores da coluna 'classe_final'.
# Isso mostra quantas instâncias existem para cada um dos possíveis resultados do jogo.
print("Distribuição inicial das classes após reclassificação")
print(df['classe_final'].value_counts())

Distribuição inicial das classes após reclassificação
classe_final
X vence    626
O vence    316
Empate      16
Name: count, dtype: int64


In [60]:
import numpy as np

# Criar 200 exemplos sintéticos de jogos que ainda estão 'Em Jogo'.
# Isso é feito para balancear o dataset, já que a classe 'Tem jogo' pode ser sub-representada.
novos_dados = []
for _ in range(200):
    tab = [0] * 9  # Inicializa um tabuleiro vazio (9 posições com 0)
    # Escolhe aleatoriamente entre 1 e 4 jogadas para simular um jogo em andamento.
    jogadas = np.random.randint(1, 5)
    # Seleciona posições aleatórias no tabuleiro onde as peças serão colocadas, sem repetição.
    posicoes = np.random.choice(9, jogadas, replace=False)
    # Preenche as posições selecionadas com 'X' (1) e 'O' (-1) alternadamente.
    for i, pos in enumerate(posicoes):
        tab[pos] = 1 if i % 2 == 0 else -1
    # Adiciona o estado do tabuleiro e a classe 'Tem jogo' à lista de novos dados.
    novos_dados.append(tab + ["Tem jogo"])

# Cria um novo DataFrame a partir dos dados sintéticos, com as mesmas colunas do 'df' original (exceto 'classe').
df_tem_jogo = pd.DataFrame(novos_dados, columns=df.columns[:-2].tolist() + ['classe_final'])
# Concatena o DataFrame original com os novos exemplos de 'Tem jogo', ignorando o índice para evitar duplicatas.
df = pd.concat([df, df_tem_jogo], ignore_index=True)

In [61]:
# Realiza o balanceamento final do dataset, garantindo que cada classe (se possível) tenha 200 amostras.
# Isso ajuda a prevenir que o modelo seja viesado.

# Cria uma lista para armazenar os DataFrames balanceados de cada grupo.
balanced_frames = []
# Agrupa o DataFrame pela coluna 'classe_final' e itera sobre cada grupo.
for name, group in df.groupby('classe_final'):
    # Amostra no máximo 200 exemplos de cada grupo. Se o grupo tiver menos de 200, ele pega todos os exemplos.
    # 'random_state=42' garante que a amostragem seja reprodutível.
    balanced_frames.append(group.sample(min(len(group), 200), random_state=42))
# Concatena todos os DataFrames amostrados em um único DataFrame balanceado e reseta o índice.
df_balanceado = pd.concat(balanced_frames).reset_index(drop=True)

# Garante que todas as colunas de posição do tabuleiro são numéricas (int).
# Define as colunas que representam as posições do tabuleiro.
board_cols = ['top_left', 'top_middle', 'top_right', 'mid_left', 'mid_middle', 'mid_right', 'bot_left', 'bot_middle', 'bot_right']
# Itera sobre cada coluna do tabuleiro.
for col in board_cols:
    df_balanceado[col] = pd.to_numeric(df_balanceado[col], errors='coerce').fillna(0).astype(int)

# Remove a coluna original 'classe' se ela ainda existir, pois 'classe_final' é a coluna alvo para o modelo.
if 'classe' in df_balanceado.columns:
    df_balanceado = df_balanceado.drop('classe', axis=1)

# Imprime a distribuição final das classes no dataset balanceado para confirmar o resultado do balanceamento.
print("Distribuição final das classes no dataset balanceado (200 amostras por classe, quando possível):")
print(df_balanceado['classe_final'].value_counts())

Distribuição final das classes no dataset balanceado (200 amostras por classe, quando possível):
classe_final
O vence     200
Tem jogo    200
X vence     200
Empate       16
Name: count, dtype: int64


In [62]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report
# Este bloco de código implementa um algoritmo de classificação de Árvore de Decisão.

# 1. Separar entrada (X) e saída (y).
# X (features/variáveis independentes): As 9 primeiras colunas, que representam o estado do tabuleiro.
X = df_balanceado.iloc[:, :9]
# y (target/variável dependente): A coluna 'classe_final', que contém o resultado do jogo.
y = df_balanceado['classe_final']

# 2. Divisão do Dataset em conjuntos de treino e teste.
# O dataset é dividido em 80% para treino e 20% para teste.
# 'test_size=0.2' especifica que 20% dos dados serão usados para teste.
# 'random_state=42' garante a reprodutibilidade da divisão.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Criar e treinar a Árvore de Decisão.
# 'DecisionTreeClassifier' é o modelo de árvore de decisão.
# 'criterion='entropy'' usa a entropia para medir a qualidade da divisão, buscando maximizar o ganho de informação.
# 'max_depth=None' permite que os nós sejam expandidos até que todas as folhas sejam puras ou contenham menos que 'min_samples_split' amostras.
modelo_arvore = DecisionTreeClassifier(criterion='entropy', max_depth=None)
modelo_arvore.fit(X_train, y_train)

# 4. Avaliação de Resultados.
# Usa o modelo treinado para fazer predições no conjunto de teste (dados que o modelo nunca viu antes).
predicoes = modelo_arvore.predict(X_test)
print("Relatório de Classificação - Árvore de Decisão:")
print(classification_report(y_test, predicoes))

Relatório de Classificação - Árvore de Decisão:
              precision    recall  f1-score   support

      Empate       0.33      0.33      0.33         3
     O vence       0.86      0.86      0.86        43
    Tem jogo       0.83      0.89      0.86        44
     X vence       0.87      0.79      0.83        34

    accuracy                           0.84       124
   macro avg       0.72      0.72      0.72       124
weighted avg       0.84      0.84      0.84       124



In [65]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Este bloco de código implementa um algoritmo de classificação Random Forest.

# 1. Criar e treinar o modelo Random Forest.
# 'n_estimators' é o número de árvores na floresta (geralmente, quanto mais, melhor, mas aumenta o tempo de computação).
# 'random_state=42' garante a reprodutibilidade.
modelo_random_forest = RandomForestClassifier(n_estimators=100, random_state=42)
modelo_random_forest.fit(X_train, y_train)

# 2. Avaliação de Resultados.
# Usa o modelo treinado para fazer predições no conjunto de teste.
predicoes_rf = modelo_random_forest.predict(X_test)
# Imprime um relatório de classificação detalhado para o modelo Random Forest.
print("Relatório de Classificação - Random Forest:")
print(classification_report(y_test, predicoes_rf))

Relatório de Classificação - Random Forest:
              precision    recall  f1-score   support

      Empate       0.00      0.00      0.00         3
     O vence       0.87      0.95      0.91        43
    Tem jogo       0.95      0.91      0.93        44
     X vence       0.86      0.88      0.87        34

    accuracy                           0.90       124
   macro avg       0.67      0.69      0.68       124
weighted avg       0.88      0.90      0.88       124



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### Salvando o Dataset Balanceado e os Modelos Treinados



In [67]:
import joblib

# 1. Salvar o dataset balanceado em CSV
output_csv_path = 'tic_tac_toe_balanceado.csv'
df_balanceado.to_csv(output_csv_path, index=False)
print(f"Dataset balanceado salvo em: {output_csv_path}")

# 2. Salvar o modelo de Árvore de Decisão
output_dt_model_path = 'modelo_arvore_decisao.joblib'
joblib.dump(modelo_arvore, output_dt_model_path)
print(f"Modelo de Árvore de Decisão salvo em: {output_dt_model_path}")

# 3. Salvar o modelo Random Forest
output_rf_model_path = 'modelo_random_forest.joblib'
joblib.dump(modelo_random_forest, output_rf_model_path)
print(f"Modelo Random Forest salvo em: {output_rf_model_path}")


Dataset balanceado salvo em: tic_tac_toe_balanceado.csv
Modelo de Árvore de Decisão salvo em: modelo_arvore_decisao.joblib
Modelo Random Forest salvo em: modelo_random_forest.joblib
